# TRT-LLM Benchmark Results — Llama 3.1 70B

Compares roofline estimator predictions against TRT-LLM gptManagerBenchmark (static batching) measurements.

Set `WORKLOAD` and `INSTANCE` below.

In [ ]:
import json
import glob
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ─── Configuration ─────────────────────────────────────────────────
WORKLOAD = "in763-out232"
INSTANCE = "g6-48xlarge"   # subdirectory name in measured/
BENCH_TOOL = "gptBench"    # gptBench or trtllm-bench
OUTPUT_LEN = 232
# ───────────────────────────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
PRED_DIR = os.path.join(BASE_DIR, WORKLOAD, "predicted")
MEAS_DIR = os.path.join(BASE_DIR, WORKLOAD, "measured", INSTANCE, BENCH_TOOL, "log")

print(f"Workload: {WORKLOAD}")
print(f"Instance: {INSTANCE}")
print(f"Predicted dir: {PRED_DIR}")
print(f"Measured dir: {MEAS_DIR}")
print(f"Predicted files: {len(glob.glob(os.path.join(PRED_DIR, 'est_*.json')))}")
print(f"Measured logs: {len(glob.glob(os.path.join(MEAS_DIR, '*.log')))}")

## 1. Parse Log Files

In [ ]:
def parse_gptbench_log(filepath: str) -> dict:
    """Parse gptManagerBenchmark log file into a dict of metrics."""
    metrics = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line.startswith("[BENCHMARK]"):
                continue
            # Format: [BENCHMARK] metric_name(unit) value
            # or:     [BENCHMARK] metric_name value
            parts = line.replace("[BENCHMARK] ", "").strip()
            # Split on last space to get key and value
            tokens = parts.rsplit(" ", 1)
            if len(tokens) != 2:
                continue
            key_raw, val_str = tokens
            # Remove unit in parentheses: avg_inter_token_latency(ms) -> avg_inter_token_latency
            key = re.sub(r'\(.*?\)', '', key_raw).strip()
            try:
                metrics[key] = float(val_str)
            except ValueError:
                metrics[key] = val_str
    return metrics


def parse_log_filename(filename: str) -> dict:
    """Extract tp, pp, bs from filename like trtllm_tp8_pp1_bs16.log"""
    m = re.match(r'trtllm_tp(\d+)_pp(\d+)_bs(\d+)\.log', filename)
    if m:
        return {"tp": int(m.group(1)), "pp": int(m.group(2)), "batch": int(m.group(3))}
    return None


def load_measured_logs(meas_dir: str) -> pd.DataFrame:
    """Load all gptManagerBenchmark log files into a DataFrame."""
    rows = []
    for filepath in sorted(glob.glob(os.path.join(meas_dir, "trtllm_*.log"))):
        filename = os.path.basename(filepath)
        info = parse_log_filename(filename)
        if info is None:
            continue
        metrics = parse_gptbench_log(filepath)
        if not metrics or metrics.get("num_samples", 0) == 0:
            continue  # skip empty logs
        
        # Compute decode_ms = e2e - ttft
        ttft = metrics.get("avg_time_to_first_token", 0)
        e2e = metrics.get("avg_sequence_latency", 0)
        itl = metrics.get("avg_inter_token_latency", 0)
        decode_ms = e2e - ttft
        
        rows.append({
            "tp": info["tp"],
            "pp": info["pp"],
            "batch": info["batch"],
            "ttft_ms": round(ttft, 2),
            "itl_ms": round(itl, 4),
            "tpot_ms": round(itl, 4),  # ITL ≈ TPOT for static batching
            "decode_ms": round(decode_ms, 2),
            "e2e_ms": round(e2e, 2),
            "rps": round(metrics.get("seq_throughput", 0), 4),
            "token_throughput": round(metrics.get("token_throughput", 0), 2),
            "num_samples": int(metrics.get("num_samples", 0)),
            "p50_ttft_ms": metrics.get("p50_time_to_first_token", 0),
            "p99_ttft_ms": metrics.get("p99_time_to_first_token", 0),
            "p50_itl_ms": metrics.get("p50_inter_token_latency", 0),
            "p99_itl_ms": metrics.get("p99_inter_token_latency", 0),
            "p50_e2e_ms": metrics.get("p50_sequence_latency", 0),
            "p99_e2e_ms": metrics.get("p99_sequence_latency", 0),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["tp", "pp", "batch"]).reset_index(drop=True)
    return df


df_meas = load_measured_logs(MEAS_DIR)
print(f"Loaded {len(df_meas)} measured entries")
df_meas

## 2. Load Estimator Predictions

In [ ]:
def load_predicted(pred_dir: str, instance_filter: str = None) -> pd.DataFrame:
    """Load estimator prediction JSONs."""
    files = sorted(glob.glob(os.path.join(pred_dir, "est_*.json")))
    rows = []
    for f in files:
        with open(f) as fp:
            d = json.load(fp)
        if not d.get("feasible", False):
            continue
        # Filter by instance if specified
        inst = d["instance_type"].replace(".", "_")
        if instance_filter and inst != instance_filter.replace("-", "_").replace(".", "_"):
            continue
        for entry in d.get("batch_sweep", []):
            rows.append({
                "instance": d["instance_type"],
                "tp": d["tp_size"],
                "pp": d["pp_size"],
                "batch": entry["batch_size"],
                "ttft_ms": round(entry.get("ttft_ms", 0), 2),
                "tpot_ms": round(entry.get("tpot_ms", 0), 4),
                "decode_ms": round(entry.get("decode_latency_ms", 0), 2),
                "e2e_ms": round(entry["batch_latency_ms"], 2),
                "rps": round(entry["throughput_rps"], 4),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["tp", "pp", "batch"]).reset_index(drop=True)
    return df


# Map INSTANCE dir name to estimator instance type
INSTANCE_MAP = {
    "g5-48xlarge": "g5.48xlarge",
    "g6-48xlarge": "g6.48xlarge",
    "g6e-48xlarge": "g6e.48xlarge",
    "p4d-24xlarge": "p4d.24xlarge",
    "p5-48xlarge": "p5.48xlarge",
}
est_instance = INSTANCE_MAP.get(INSTANCE, INSTANCE)

df_pred = load_predicted(PRED_DIR, INSTANCE)
print(f"Loaded {len(df_pred)} predicted entries for {est_instance}")
df_pred

## 3. Merge & Compute MAPE

In [ ]:
COMPARE_COLS = ["ttft_ms", "tpot_ms", "decode_ms", "e2e_ms", "rps"]

def merge_and_compare(df_pred: pd.DataFrame, df_meas: pd.DataFrame) -> pd.DataFrame:
    """Merge predicted and measured on (tp, pp, batch), compute MAPE."""
    merged = pd.merge(
        df_pred, df_meas,
        on=["tp", "pp", "batch"],
        suffixes=("_est", "_meas"),
        how="inner"
    )
    for col in COMPARE_COLS:
        est_col = f"{col}_est"
        meas_col = f"{col}_meas"
        if est_col in merged.columns and meas_col in merged.columns:
            merged[f"{col}_mape%"] = (
                (merged[est_col] - merged[meas_col]).abs() / merged[meas_col].abs() * 100
            ).round(1)
    return merged


df_cmp = merge_and_compare(df_pred, df_meas)
print(f"Matched {len(df_cmp)} (tp, pp, batch) entries")

# Display key columns
display_cols = ["tp", "pp", "batch"]
for col in COMPARE_COLS:
    for suffix in ["_est", "_meas", "_mape%"]:
        c = f"{col}{suffix}"
        if c in df_cmp.columns:
            display_cols.append(c)

df_cmp[display_cols]

## 4. Summary: Average MAPE by Strategy

In [ ]:
if not df_cmp.empty:
    mape_cols = [c for c in df_cmp.columns if c.endswith("_mape%")]
    summary = df_cmp.groupby(["tp", "pp"])[mape_cols].mean().round(1)
    print("Average MAPE (%) by strategy:")
    display(summary)
    
    print("\nOverall MAPE (%):")
    print(df_cmp[mape_cols].mean().round(1).to_string())

## 5. Charts: Estimated vs Measured by Batch Size

In [ ]:
def plot_est_vs_meas(df: pd.DataFrame, metric: str, ylabel: str, title_suffix: str = ""):
    """Plot estimated vs measured for a metric across batch sizes, one subplot per strategy."""
    strategies = df.groupby(["tp", "pp"]).ngroups
    if strategies == 0:
        return
    
    fig, axes = plt.subplots(1, strategies, figsize=(6 * strategies, 5), squeeze=False)
    axes = axes.flatten()
    
    for idx, ((tp, pp), group) in enumerate(df.groupby(["tp", "pp"])):
        ax = axes[idx]
        g = group.sort_values("batch")
        
        est_col = f"{metric}_est"
        meas_col = f"{metric}_meas"
        
        if est_col not in g.columns or meas_col not in g.columns:
            continue
        
        x = np.arange(len(g))
        w = 0.35
        
        bars_est = ax.bar(x - w/2, g[est_col], w, label="Estimated", color="steelblue", alpha=0.8)
        bars_meas = ax.bar(x + w/2, g[meas_col], w, label="Measured", color="coral", alpha=0.8)
        
        # Add MAPE annotations
        mape_col = f"{metric}_mape%"
        if mape_col in g.columns:
            for i, (_, row) in enumerate(g.iterrows()):
                max_val = max(row[est_col], row[meas_col])
                ax.text(i, max_val * 1.02, f"{row[mape_col]:.0f}%",
                        ha='center', va='bottom', fontsize=8, color='gray')
        
        ax.set_xlabel("Batch Size")
        ax.set_ylabel(ylabel)
        ax.set_title(f"tp{tp}_pp{pp} — {INSTANCE}")
        ax.set_xticks(x)
        ax.set_xticklabels(g["batch"].astype(str), rotation=45)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    fig.suptitle(f"{ylabel} — Estimated vs Measured (TRT-LLM){title_suffix}", fontsize=13)
    plt.tight_layout()
    plt.show()


if not df_cmp.empty:
    plot_est_vs_meas(df_cmp, "e2e_ms", "E2E Latency (ms)")
    plot_est_vs_meas(df_cmp, "ttft_ms", "TTFT (ms)")
    plot_est_vs_meas(df_cmp, "decode_ms", "Decode Latency (ms)")
    plot_est_vs_meas(df_cmp, "tpot_ms", "TPOT / ITL (ms)")
    plot_est_vs_meas(df_cmp, "rps", "Throughput (req/sec)")

## 6. Scaling Analysis: Batch Size vs Decode Latency

In [ ]:
def plot_scaling(df: pd.DataFrame, metric: str, ylabel: str):
    """Line plot showing how a metric scales with batch size."""
    strategies = df.groupby(["tp", "pp"]).ngroups
    if strategies == 0:
        return
    
    fig, axes = plt.subplots(1, strategies, figsize=(6 * strategies, 5), squeeze=False)
    axes = axes.flatten()
    
    for idx, ((tp, pp), group) in enumerate(df.groupby(["tp", "pp"])):
        ax = axes[idx]
        g = group.sort_values("batch")
        
        est_col = f"{metric}_est"
        meas_col = f"{metric}_meas"
        
        if est_col in g.columns:
            ax.plot(g["batch"], g[est_col], 'o-', label="Estimated", color="steelblue")
        if meas_col in g.columns:
            ax.plot(g["batch"], g[meas_col], 's-', label="Measured", color="coral")
        
        ax.set_xlabel("Batch Size")
        ax.set_ylabel(ylabel)
        ax.set_title(f"tp{tp}_pp{pp} — {INSTANCE}")
        ax.set_xscale("log", base=2)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    fig.suptitle(f"{ylabel} Scaling — Estimated vs Measured", fontsize=13)
    plt.tight_layout()
    plt.show()


if not df_cmp.empty:
    plot_scaling(df_cmp, "decode_ms", "Decode Latency (ms)")
    plot_scaling(df_cmp, "tpot_ms", "TPOT / ITL (ms)")
    plot_scaling(df_cmp, "ttft_ms", "TTFT (ms)")

## 7. Raw Measured Data (All Metrics)

In [ ]:
print(f"All measured results for {INSTANCE}:")
df_meas

## 8. Decode Scaling Ratio (bs=N / bs=1)

In [ ]:
if not df_cmp.empty:
    for (tp, pp), group in df_cmp.groupby(["tp", "pp"]):
        g = group.sort_values("batch")
        bs1_est = g.loc[g["batch"] == 1, "decode_ms_est"]
        bs1_meas = g.loc[g["batch"] == 1, "decode_ms_meas"]
        
        if bs1_est.empty or bs1_meas.empty:
            continue
        
        bs1_est_val = bs1_est.values[0]
        bs1_meas_val = bs1_meas.values[0]
        
        print(f"\ntp{tp}_pp{pp} — Decode scaling ratio (vs bs=1):")
        print(f"{'Batch':>6} {'Est ratio':>10} {'Meas ratio':>11} {'Est(ms)':>10} {'Meas(ms)':>10}")
        print("-" * 52)
        for _, row in g.iterrows():
            bs = int(row["batch"])
            est_ratio = row["decode_ms_est"] / bs1_est_val
            meas_ratio = row["decode_ms_meas"] / bs1_meas_val
            print(f"{bs:>6} {est_ratio:>10.2f}x {meas_ratio:>10.2f}x {row['decode_ms_est']:>10.1f} {row['decode_ms_meas']:>10.1f}")